# CS-781 Phase 7 (Prior Case Retrieval) - `p7_fusion`

No training, **CPU is enough** (accelerator None), Internet on, secrets `HF` and `HF_WRITE`. Loads the dev / test similarity matrices of every finished setup from the bucket and compares: TF-IDF and BM25 alone and fused; each trained setup + lambda x TF-IDF on a finer lambda grid; reciprocal-rank fusion; ensembles of the Phase-1 dense models alone (top-k and greedy); dense ensembles + TF-IDF; paired bootstrap intervals vs TF-IDF. Writes `submission_p7_dense_ensemble.zip` (Phase-1 models only) and `submission_p7_fusion.zip` (best combination with TF-IDF). Setups that have not run yet are skipped. About 10 minutes.

In [1]:
TOKENIZER_NAME = None
!pip install -q -U "huggingface_hub>=1.5"
import os, json, shutil, subprocess, sys, time, zipfile, random, math, gc
from collections import Counter, defaultdict
import torch, torch.nn as nn, torch.nn.functional as F
from huggingface_hub import hf_hub_download, login, download_bucket_files, batch_bucket_files
from kaggle_secrets import UserSecretsClient

NB_START = time.time()
WORK = "/kaggle/working"
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
HF_BUCKET = "kavincg23/cs781-models"
HF_CHECKPOINT_REPO = "kavincg23/cs781-checkpoints"   # old repo, read-only fallback
user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF")
HF_WRITE_TOKEN = user_secrets.get_secret("HF_WRITE")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, torch.cuda.get_device_name(0) if device.type == "cuda" else "")


def get_dataset_file(filename):
    return hf_hub_download(repo_id=HF_DATASET_REPO, filename=filename, repo_type="dataset", token=HF_TOKEN)


def fetch_model_file(name, local_path=None):
    local_path = local_path or f"{WORK}/{name}"
    if os.path.exists(local_path):
        return local_path
    part = local_path + ".part"
    try:
        download_bucket_files(HF_BUCKET, files=[(name, part)], raise_on_missing_files=True, token=HF_WRITE_TOKEN)
        os.replace(part, local_path)
        return local_path
    except Exception:
        if os.path.exists(part):
            os.remove(part)
    # fallback: the OLD model repo (the Phase-1 notebooks do the same; the tokenizer only lives there)
    p = hf_hub_download(repo_id=HF_CHECKPOINT_REPO, filename=name, repo_type="model", token=HF_WRITE_TOKEN)
    shutil.copy(p, local_path)
    print(f"(found {name} in the old repo {HF_CHECKPOINT_REPO})")
    return local_path


def upload_model_file(local_path, name=None):
    name = name or os.path.basename(local_path)
    try:
        batch_bucket_files(HF_BUCKET, add=[(local_path, name)], token=HF_WRITE_TOKEN)
        print(f"uploaded {name} to bucket {HF_BUCKET}")
    except Exception as e:
        print(f"(upload of {name} failed: {type(e).__name__}: {e}; the file is still in {WORK})")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 2.4.0 which is incompatible.
gradio 6.26.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
datasets 4.8.5 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 2.4.0 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 2.4.0 which is incompatible.
transformers 5.16.1 requires huggingface-hub<2.0,>=1.5.0, but you have huggingface-hub 2.4.0 which is incompatible.
gradio-client 2.6.1 requires huggingface-hub<2.0,>=0.19.3, but you have huggingface-hub 2.4.0 which is incompatible.
tokenizers 0.23.1 requires huggingface-hub<2.0,>=0.16.4, but you have huggingface-hub 2.4.0 which is incompatible.


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


device: cuda Tesla T4


In [2]:
# ======================================================================================================================
# Phase 7 (Prior Case Retrieval) engine: case text (with / without the citation header), chunked case encoder built on a
# Phase-1 backbone, query-candidate similarity, contrastive / pairwise / listwise objectives with in-batch and hard
# negatives, knowledge distillation (teacher similarity scores or embeddings), Micro-F1@7 evaluation, TF-IDF hybrid,
# submission files. Everything model-specific comes from the Phase-1 / Phase-2 cells above (backbone.features()).
# ======================================================================================================================
import math, time, random, json, os, re, zipfile, collections, hashlib
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint
K_EVAL = 7


# ----------------------------------------------------------------------------------------------------------------------
# 1. Case text; the Supreme-Court header (CITATION / CITATOR INFO) leaks citation links -> HEADER="strip" removes them
# ----------------------------------------------------------------------------------------------------------------------
def doc_text(v):
    if isinstance(v, dict):
        for k in ("text", "sentences", "document", "content"):
            if k in v: return doc_text(v[k])
        return ""
    if isinstance(v, list):
        return " ".join(x if isinstance(x, str) else doc_text(x) for x in v)
    return str(v)


_HDR_BLOCK = re.compile(r"(CITATION\s*:.*?|CITATOR\s+INFO\s*:.*?)(?=\bACT\s*:|\bHEADNOTE\s*:|\bJUDGMENT\b|\bCITATOR\s+INFO\b|$)", re.S)
_REPORTER = re.compile(r"\(?\b(?:19|20)\d{2}\)?\s*(?:\(\s*\d+\s*\)\s*)?(?:AIR|SCR|SCC|SCALE|JT|SC|Cri\s*LJ|CriLJ|ITR|SCJ|MLJ|Bom\s*LR|ALL|Cal|Mad|Bom|Ker|Pat|Del)\b"
                       r"\.?\s*(?:\(\s*\d+\s*\)\s*)?(?:Supl\.?|Supp\.?|SC)?\s*\(?\d*\)?\s*\d{1,5}(?:\s*\(\s*[\d,\s]+\))?")
_REPORTER2 = re.compile(r"\bAIR\s+(?:19|20)\d{2}\s+[A-Za-z]{2,6}\s+\d{1,5}")
_REPORTER4 = re.compile(r"\b(?:19|20)\d{2}\s+Supp?l?\.?\s*(?:\(\s*\d+\s*\)\s*)?(?:SCR|SCC)\s*\d{1,5}")
_REPORTER3 = re.compile(r"[\(\[]\s*(?:19|20)\d{2}\s*[\)\]]\s*\d{0,3}\s*(?:SCC|SCR|SCALE|JT|SC|ITR|Cri\s*LJ|CriLJ|SCJ|MLJ)\b\.?\s*(?:\(\s*\w+\s*\)\s*)?\d{1,5}")


def strip_header(t):
    """Remove the citation strings that link cases: the CITATION / CITATOR INFO blocks of the Supreme-Court header (only
    in the first 4,000 characters) and every reporter citation in the body (e.g. '1964 AIR 1545', 'AIR 1954 SC 210',
    '(1990) 1 SCC 445')."""
    head, body = t[:4000], t[4000:]
    head = _HDR_BLOCK.sub(" ", head)
    t = head + body
    t = _REPORTER2.sub(" <CITATION> ", t)
    t = _REPORTER3.sub(" <CITATION> ", t)
    t = _REPORTER4.sub(" <CITATION> ", t)
    t = _REPORTER.sub(" <CITATION> ", t)
    return re.sub(r"\s+", " ", t)


def case_text(v, mode):
    t = doc_text(v)
    return strip_header(t) if mode == "strip" else t


_OWN = [re.compile(r"\b((?:19|20)\d{2})\s+AIR\s+(\d{1,5})"), re.compile(r"\bAIR\s+((?:19|20)\d{2})\s+SC\s*(\d{1,5})")]
_CITATOR_REF = re.compile(r"\b((?:19|20)\d{2})\s*SC\s*(\d{1,5})")


def citation_keys(t):
    """(own AIR citations, AIR citations listed in its CITATOR INFO = later judgments that cite it) from the header."""
    h = t[:5000]; own, refs = set(), set()
    m = re.search(r"CITATION\s*:(.*?)(?:CITATOR\s+INFO|ACT\s*:|HEADNOTE|$)", h, re.S)
    if m:
        for p in _OWN: own |= {(a, str(int(b))) for a, b in p.findall(m.group(1))}
    m = re.search(r"CITATOR\s+INFO\s*:(.*?)(?:ACT\s*:|HEADNOTE|JUDGMENT|$)", h, re.S)
    if m:
        refs = {(a, str(int(b))) for a, b in _CITATOR_REF.findall(m.group(1))}
    return own, refs


def encode_case(tok, text, max_tokens):
    ids = tok.encode(text)
    if len(ids) > max_tokens:
        h = max_tokens // 2; ids = ids[:h] + ids[-(max_tokens - h):]
    return np.asarray(ids, dtype=np.uint16)


# ----------------------------------------------------------------------------------------------------------------------
# 2. Chunks and batches
# ----------------------------------------------------------------------------------------------------------------------
def make_spans(n, W, S, m, how, head_chunks):
    if n <= W:
        return [(0, max(n, 1))]
    starts = list(range(0, n - W + 1, max(1, S)))
    if starts[-1] + W < n: starts.append(n - W)
    spans = [(s, s + W) for s in starts]; K = len(spans)
    if m is None or K <= m or how == "all":
        return spans
    if how == "first": idx = list(range(m))
    elif how == "last": idx = list(range(K - m, K))
    elif how == "head_tail":
        h = min(head_chunks, m - 1) if m > 1 else 0; idx = list(range(h)) + list(range(K - (m - h), K))
    elif how == "uniform": idx = sorted(set(int(round(x)) for x in np.linspace(0, K - 1, m)))
    else: raise ValueError(how)
    return [spans[i] for i in idx]


def collate_docs(token_lists, C, n_chunks, pad_id=0):
    rows, counts = [], []
    for ids in token_lists:
        sp = make_spans(len(ids), C["WINDOW"], C["STRIDE"], n_chunks, C["SELECT"], C["HEAD_CHUNKS"])
        for s, e in sp:
            r = ids[s:e]; rows.append(r if len(r) else np.zeros(1, dtype=np.uint16))
        counts.append(len(sp))
    R, T = len(rows), max(len(r) for r in rows)
    X = torch.full((R, T), pad_id, dtype=torch.long); M = torch.zeros((R, T), dtype=torch.bool)
    for r, a in enumerate(rows):
        X[r, :len(a)] = torch.from_numpy(a.astype(np.int64)); M[r, :len(a)] = True
    K = max(counts)
    cm = torch.arange(K).unsqueeze(0) < torch.tensor(counts).unsqueeze(1)
    return dict(ids=X, mask=M, chunk_mask=cm)


def to_device(b, device):
    return {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in b.items()}


# ----------------------------------------------------------------------------------------------------------------------
# 3. Case encoder: Phase-1 backbone -> chunk vectors (token pooling) -> case vector (chunk pooling) -> projection
# ----------------------------------------------------------------------------------------------------------------------
class CaseEncoder(nn.Module):
    """Attribute names shared with the Phase-3/5 model (backbone, soft_prompt, head_parameters) so the technique code works unchanged.
    POOL (tokens of a chunk): mean | last | mean+last | max.  DOC_POOL (chunks of a case): mean | max | attn | first | maxsim
    (maxsim = late interaction: the case is kept as its chunk vectors; score = mean over query chunks of the best candidate chunk)."""

    def __init__(self, backbone, d_model, C):
        super().__init__()
        self.backbone, self.soft_prompt = backbone, None
        self.pool, self.doc_pool, self.sim = C["POOL"], C["DOC_POOL"], C["SIM"]
        self.row_group, self.ckpt = C["ROW_GROUP"], C.get("CKPT", True)
        if self.ckpt and any(getattr(m, "bias_rate", 0) > 0 or (hasattr(m, "tau") and getattr(type(m)._route, "__name__", "") != "_mor_route_frozen") for m in backbone.modules()):
            self.ckpt = False
        d = d_model * (2 if self.pool == "mean+last" else 1)
        self.attn = nn.Linear(d, 1) if self.doc_pool == "attn" else None
        self.proj = nn.Linear(d, C["PROJ_DIM"]) if C["PROJ_DIM"] else nn.Identity()
        if C["PROJ_DIM"] and C.get("PROJ_INIT", "random") == "random":
            nn.init.normal_(self.proj.weight, std=1.0 / math.sqrt(d)); nn.init.zeros_(self.proj.bias)
        self.out_dim = C["PROJ_DIM"] or d

    def head_parameters(self):
        return list(self.proj.parameters()) + (list(self.attn.parameters()) if self.attn is not None else [])

    def _pool_rows(self, ids, mask):
        h = self.backbone.features(ids, mask, self.soft_prompt).float()
        m = mask.unsqueeze(-1).float(); parts = []
        if self.pool in ("mean", "mean+last"):
            parts.append((h * m).sum(1) / m.sum(1).clamp(min=1))
        if self.pool in ("last", "mean+last"):
            parts.append(h[torch.arange(h.shape[0], device=h.device), mask.sum(1) - 1])
        if self.pool == "max":
            parts.append(h.masked_fill(~mask.unsqueeze(-1), -1e4).max(1).values)
        return torch.cat(parts, -1)

    def chunk_vectors(self, b):
        ids, mask = b["ids"], b["mask"]
        trainable = self.training and (any(p.requires_grad for p in self.backbone.parameters()) or self.soft_prompt is not None)
        out = []
        for a in range(0, ids.shape[0], self.row_group):
            i, m = ids[a:a + self.row_group], mask[a:a + self.row_group]
            L = int(m.sum(1).max()); i, m = i[:, :L], m[:, :L]
            if not trainable:
                with torch.no_grad(): out.append(self._pool_rows(i, m))
            elif self.ckpt: out.append(checkpoint(self._pool_rows, i, m, use_reentrant=False))
            else: out.append(self._pool_rows(i, m))
        return torch.cat(out, 0)

    def forward(self, b):
        """-> (B, D) case vectors, or (B, K, D) chunk vectors + (B, K) mask for maxsim."""
        v = self.chunk_vectors(b); cm = b["chunk_mask"]
        vp = v.new_zeros(cm.shape[0], cm.shape[1], v.shape[1]); vp[cm] = v
        if self.doc_pool == "maxsim":
            z = self.proj(vp)
            return (F.normalize(z, dim=-1) if self.sim == "cos" else z), cm
        m = cm.unsqueeze(-1).float()
        if self.doc_pool == "mean": d = (vp * m).sum(1) / m.sum(1).clamp(min=1)
        elif self.doc_pool == "max": d = vp.masked_fill(~cm.unsqueeze(-1), -1e4).max(1).values
        elif self.doc_pool == "first": d = vp[:, 0]
        elif self.doc_pool == "attn":
            a = self.attn(vp).squeeze(-1).masked_fill(~cm, -1e4); d = (torch.softmax(a, -1).unsqueeze(-1) * vp).sum(1)
        else: raise ValueError(self.doc_pool)
        z = self.proj(d)
        return F.normalize(z, dim=-1) if self.sim == "cos" else z


def sim_matrix(Q, Cn, kind):
    """Q, Cn: case vectors (B, D) / (N, D), or (vecs, mask) pairs for maxsim."""
    if isinstance(Q, tuple):
        (q, qm), (c, cmask) = Q, Cn
        s = torch.einsum("aid,bjd->abij", q, c)                                   # (B, N, Kq, Kc)
        s = s.masked_fill(~cmask[None, :, None, :], -1e4).max(-1).values          # best candidate chunk
        qmf = qm.float()[:, None, :]
        return (s * qmf).sum(-1) / qmf.sum(-1).clamp(min=1)
    s = Q @ Cn.T
    return s if kind == "cos" else s / math.sqrt(Q.shape[1])


# ----------------------------------------------------------------------------------------------------------------------
# 4. Training examples, objectives, KD
# ----------------------------------------------------------------------------------------------------------------------
def sample_batch(qids, rel, pool, hard, C, rng):
    """For each query: LIST_POS positives (InfoNCE / pairwise use the first) and N_NEG negatives (random / hard / mixed)."""
    pos_lists, neg_lists = [], []
    for q in qids:
        r = [c for c in rel[q] if c != q and c in pool]
        pos_lists.append(rng.sample(r, min(len(r), C["LIST_POS"] if C["OBJECTIVE"] == "listwise" else 1)))
        bad = set(rel[q]) | {q}; negs = []
        for k in range(C["N_NEG"]):
            how = C["NEG"] if C["NEG"] != "mixed" else ("hard" if k % 2 == 0 else "random")
            if how == "hard" and hard.get(q):
                cand = [c for c in hard[q][:C["HARD_TOP"]] if c not in bad and c not in negs]
            else:
                cand = None
            if cand:
                negs.append(rng.choice(cand))
            else:
                for _ in range(50):
                    c = rng.choice(pool)
                    if c not in bad and c not in negs: negs.append(c); break
        neg_lists.append(negs)
    return pos_lists, neg_lists


def retrieval_loss(S, B, cand_ids, qids, pos_lists, neg_lists, rel, C):
    """S: (B, N) query x candidate scores for this batch (candidates = every positive and negative of every query, in order)."""
    tau = C["TAU"]; dev = S.device
    relset = [set(rel[q]) | {q} for q in qids]
    pos_mask = torch.tensor([[c in relset[i] and c != qids[i] for c in cand_ids] for i in range(B)], device=dev)
    own = [set(p) | set(n) for p, n in zip(pos_lists, neg_lists)]
    if C["OBJECTIVE"] in ("infonce", "infonce_own"):
        tgt = torch.tensor([cand_ids.index(p[0]) for p in pos_lists], device=dev)
        keep = torch.ones_like(S, dtype=torch.bool)
        keep &= ~(pos_mask & (torch.arange(S.shape[1], device=dev)[None] != tgt[:, None]))       # other positives are not negatives
        if C["OBJECTIVE"] == "infonce_own":                                                        # no in-batch negatives
            keep &= torch.tensor([[c in own[i] for c in cand_ids] for i in range(B)], device=dev)
        return F.cross_entropy((S / tau).masked_fill(~keep, -1e4), tgt)
    if C["OBJECTIVE"] == "listwise":                                                               # ListNet: target = uniform over all positives in the batch list
        t = pos_mask.float(); t = t / t.sum(1, keepdim=True).clamp(min=1)
        return -(t * F.log_softmax(S / tau, -1)).sum(1).mean()
    if C["OBJECTIVE"] in ("pairwise", "margin"):                                                   # own positive vs each own negative
        terms = []
        for i in range(B):
            p = cand_ids.index(pos_lists[i][0])
            for n in neg_lists[i]:
                d = S[i, cand_ids.index(n)] - S[i, p]
                terms.append(F.softplus(d / tau) if C["OBJECTIVE"] == "pairwise" else F.relu(C["MARGIN"] + d))
        return torch.stack(terms).mean()
    raise ValueError(C["OBJECTIVE"])


def kd_terms(S, qv, cv, qids, cand_ids, teacher, C):
    """KD_MODE scores: KL(teacher || student) over the batch's candidate list (teacher similarity distribution);
    embed: cosine distance between student and teacher case vectors (intermediate representation, PROJ_DIM must match)."""
    out = 0.0
    TQ = torch.stack([teacher[q] for q in qids]).to(S.device).float(); TC = torch.stack([teacher[c] for c in cand_ids]).to(S.device).float()
    if C["KD_MODE"] in ("scores", "both"):
        St = TQ @ TC.T; T = C["KD_TEMPERATURE"]
        out = out + F.kl_div(F.log_softmax(S / C["TAU"] / T, -1), F.softmax(St / C["TEACHER_TAU"] / T, -1), reduction="batchmean") * T * T
    if C["KD_MODE"] in ("embed", "both"):
        assert qv.shape[-1] == TQ.shape[-1], "embedding KD needs PROJ_DIM equal to the teacher's"
        out = out + (1 - F.cosine_similarity(qv, TQ, -1)).mean() + (1 - F.cosine_similarity(cv, TC, -1)).mean()
    return out


# ----------------------------------------------------------------------------------------------------------------------
# 5. Encoding, ranking, scores
# ----------------------------------------------------------------------------------------------------------------------
@torch.no_grad()
def encode_cases(model, toks, ids, C, device, n_chunks, bs=None):
    """-> {id: vector (D,) or (chunk vectors, mask)} on CPU, fp16."""
    model.eval(); out = {}; bs = bs or C["INFER_BATCH"]
    use_amp = device.type == "cuda" and C["AMP"] == "fp16"
    order = sorted(ids, key=lambda i: len(toks[i]))
    for a in range(0, len(order), bs):
        grp = order[a:a + bs]
        b = to_device(collate_docs([toks[i] for i in grp], C, n_chunks), device)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            v = model(b)
        if isinstance(v, tuple):
            for k, i in enumerate(grp): out[i] = (v[0][k].half().cpu(), v[1][k].cpu())
        else:
            for k, i in enumerate(grp): out[i] = v[k].float().half().cpu()
    return out


def score_matrix(E, qids, cids, kind, device, chunk=256):
    if isinstance(E[qids[0]], tuple):
        Kq = max(E[q][0].shape[0] for q in qids); Kc = max(E[c][0].shape[0] for c in cids)
        def stack(ids, K):
            v = torch.zeros(len(ids), K, E[ids[0]][0].shape[-1]); m = torch.zeros(len(ids), K, dtype=torch.bool)
            for i, x in enumerate(ids): k = E[x][0].shape[0]; v[i, :k] = E[x][0].float(); m[i, :k] = E[x][1][:k]
            return v.to(device), m.to(device)
        Cv = stack(cids, Kc); rows = []
        for a in range(0, len(qids), 16):
            rows.append(sim_matrix(stack(qids[a:a + 16], Kq), Cv, kind).cpu())
        return torch.cat(rows).numpy()
    Q = torch.stack([E[q].float() for q in qids]).to(device); Cm = torch.stack([E[c].float() for c in cids]).to(device)
    return torch.cat([sim_matrix(Q[a:a + chunk], Cm, kind).cpu() for a in range(0, len(qids), chunk)]).numpy()


def rank_from_scores(S, qids, cids, k=200):
    pos = {c: j for j, c in enumerate(cids)}; S = S.copy()
    for i, q in enumerate(qids):
        if q in pos: S[i, pos[q]] = -1e9                                       # a case is never its own prior case
    top = np.argsort(-S, 1)[:, :k]
    return {q: [cids[j] for j in top[i]] for i, q in enumerate(qids)}


def prf_at(gold, ranked, k=K_EVAL):
    tp = fp = fn = 0
    for q, g in gold.items():
        g = set(g); p = set(ranked[q][:k]); tp += len(g & p); fp += len(p - g); fn += len(g - p)
    return dict(f1=2 * tp / max(1, 2 * tp + fp + fn), p=tp / max(1, tp + fp), r=tp / max(1, tp + fn), tp=tp, fp=fp, fn=fn)


def retrieval_scores(gold, ranked):
    m = prf_at(gold, ranked)
    tot = sum(len(g) for g in gold.values())
    for k in (20, 50, 100, 200):
        m[f"r@{k}"] = sum(len(set(g) & set(ranked[q][:k])) for q, g in gold.items()) / max(1, tot)
    rr = []
    for q, g in gold.items():
        g = set(g); r = next((i + 1 for i, c in enumerate(ranked[q]) if c in g), None); rr.append(1 / r if r else 0.0)
    m["mrr"] = float(np.mean(rr)) if rr else 0.0
    return m


def zscore_rows(S):
    return (S - S.mean(1, keepdims=True)) / (S.std(1, keepdims=True) + 1e-9)


def hybrid_scores(Sd, St, lam):
    return zscore_rows(Sd) + lam * zscore_rows(St)


# ----------------------------------------------------------------------------------------------------------------------
# 6. Training loop
# ----------------------------------------------------------------------------------------------------------------------
def _lr_lambda(total, warmup):
    def f(step):
        if step < warmup: return (step + 1) / max(1, warmup)
        return max(0.0, (total - step) / max(1, total - warmup))
    return f


def trainable_state(model):
    st = {n: p.detach().to("cpu", copy=True) for n, p in model.named_parameters() if p.requires_grad}
    for n, bf in model.named_buffers():
        if n.endswith((".tau", ".tau_ready", ".ada_mask")): st[n] = bf.detach().to("cpu", copy=True)
    return st


def load_trainable_state(model, st):
    known = dict(model.named_parameters()); known.update(dict(model.named_buffers()))
    missing = [k for k in st if k not in known]
    assert not missing, f"state has unknown keys: {missing[:5]}"
    model.load_state_dict(st, strict=False)


def train_retriever(model, data, C, device, evaluate, log=print, teacher=None):
    """data: dict(qids (training queries with >=1 relevant), rel, pool, toks, hard). evaluate(model) -> dict with 'f1'.
    An epoch = every training query once (shuffled), BATCH queries per step."""
    rng = random.Random(C["SEED"]); torch.manual_seed(C["SEED"])
    params_bb = [p for n, p in model.named_parameters() if p.requires_grad and not n.startswith(("proj.", "attn."))]
    params_hd = [p for n, p in model.named_parameters() if p.requires_grad and n.startswith(("proj.", "attn."))]
    groups = [g for g in (dict(params=params_bb, lr=C["LR"]), dict(params=params_hd, lr=C["HEAD_LR"])) if g["params"]]
    opt = torch.optim.AdamW(groups, weight_decay=C["WEIGHT_DECAY"], betas=(0.9, 0.98))
    qids = list(data["qids"]); steps_ep = math.ceil(len(qids) / C["BATCH"]); total = steps_ep * C["EPOCHS"]
    sched = torch.optim.lr_scheduler.LambdaLR(opt, _lr_lambda(total, int(C["WARMUP_FRAC"] * total)))
    use_amp = device.type == "cuda" and C["AMP"] == "fp16"; scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    if device.type == "cuda": torch.cuda.reset_peak_memory_stats(device)
    hist, best, best_state, best_ep = [], -1.0, None, 0
    allp = [p for g in groups for p in g["params"]]
    t0 = time.time(); n_docs = 0
    log(f"training: {len(qids)} queries per epoch, {C['BATCH']} per step ({steps_ep} steps), {C['EPOCHS']} epochs; objective {C['OBJECTIVE']}, {C['N_NEG']} {C['NEG']} negative(s) per query"
        + (" + in-batch" if C["OBJECTIVE"] in ("infonce", "listwise") else "") + f"; {C['TRAIN_CHUNKS']} chunks per case in training, {C['INFER_CHUNKS']} at inference; "
        + f"{sum(p.numel() for p in allp):,} trainable parameters" + (f"; KD ({C['KD_MODE']}, alpha {C['KD_ALPHA']})" if teacher is not None else ""))
    r0 = None
    if C.get("EVAL_EPOCH0"):      # the Phase-1 representation as it is (+ the untrained projection): how well does it transfer without training?
        r0 = evaluate(model); log(f"  epoch 0 (before training, Phase-1 weights as they are): dev F1@7 {r0['f1']:.4f} R@100 {r0['r@100']:.3f} MRR {r0['mrr']:.4f}")
    for ep in range(1, C["EPOCHS"] + 1):
        model.train(); rng.shuffle(qids); run, nb, t_ep = 0.0, 0, time.time()
        for a in range(0, len(qids), C["BATCH"]):
            bq = qids[a:a + C["BATCH"]]
            pos_l, neg_l = sample_batch(bq, data["rel"], data["pool"], data["hard"], C, rng)
            cand = list(dict.fromkeys([c for p, n in zip(pos_l, neg_l) for c in p + n]))
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                bqv = to_device(collate_docs([data["toks"][q] for q in bq], C, C["TRAIN_CHUNKS"]), device)
                bcv = to_device(collate_docs([data["toks"][c] for c in cand], C, C["TRAIN_CHUNKS"]), device)
                qv, cv = model(bqv), model(bcv)
                S = sim_matrix(qv, cv, C["SIM"]).float()
                loss = retrieval_loss(S, len(bq), cand, bq, pos_l, neg_l, data["rel"], C)
                if teacher is not None and C["KD_ALPHA"] > 0:
                    loss = (1 - C["KD_ALPHA"]) * loss + C["KD_ALPHA"] * kd_terms(S, qv if not isinstance(qv, tuple) else qv[0].mean(1),
                                                                                cv if not isinstance(cv, tuple) else cv[0].mean(1), bq, cand, teacher, C)
                if hasattr(model, "extra_loss"): loss = loss + model.extra_loss()
            n_docs += len(bq) + len(cand)
            if not torch.isfinite(loss): continue
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.unscale_(opt)
            if hasattr(model, "on_before_step"): model.on_before_step(len(hist) * steps_ep + nb, total)
            torch.nn.utils.clip_grad_norm_(allp, C["GRAD_CLIP"]); scaler.step(opt); scaler.update(); sched.step()
            run += loss.item(); nb += 1
        t_tr = time.time() - t_ep; t_e = time.time(); r = evaluate(model)
        row = dict(epoch=ep, train_loss=round(run / max(1, nb), 4), dev_f1=round(r["f1"], 4), dev_p=round(r["p"], 4), dev_r=round(r["r"], 4),
                   dev_r100=round(r["r@100"], 3), dev_mrr=round(r["mrr"], 4), train_min=round(t_tr / 60, 1), eval_min=round((time.time() - t_e) / 60, 1))
        hist.append(row); log("  epoch %d: " % ep + ", ".join(f"{k}={v}" for k, v in row.items() if k != "epoch"))
        if r["f1"] > best:
            best, best_ep, best_state = r["f1"], ep, trainable_state(model)
        if C.get("TIME_BUDGET_MIN") and (time.time() - t0) / 60 > C["TIME_BUDGET_MIN"]:
            log(f"  time budget reached after epoch {ep}"); break
    train_secs = sum(h["train_min"] for h in hist) * 60
    peak = torch.cuda.max_memory_allocated(device) / 1e9 if device.type == "cuda" else None
    return dict(epoch0=r0, history=hist, best_epoch=best_ep, best_dev_f1=best, best_state=best_state, train_seconds=round(train_secs, 1),
                peak_mem_gb=round(peak, 3) if peak else None, docs_per_sec=round(n_docs / max(train_secs, 1e-9), 2))


# ----------------------------------------------------------------------------------------------------------------------
# 7. Submission: prediction.json = {query_id: [7 candidate ids, best first]} in a zip
# ----------------------------------------------------------------------------------------------------------------------
def write_submission(ranked, qids, pool, out_dir, name):
    pred = {}
    for q in qids:
        lst = [c for c in ranked[q] if c != q and c in pool]
        lst = list(dict.fromkeys(lst))[:K_EVAL]
        assert len(lst) == K_EVAL, f"query {q}: fewer than {K_EVAL} candidates"
        pred[q] = lst
    os.makedirs(out_dir, exist_ok=True)
    jp = os.path.join(out_dir, f"{name}_prediction.json"); json.dump(pred, open(jp, "w"))
    errs = validate(json.load(open(jp)), {q: [] for q in qids}, set(pool))           # the course evaluator's checks
    assert not errs, errs[:5]
    zp = os.path.join(out_dir, f"submission_{name}.zip")
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z: z.write(jp, arcname="prediction.json")
    return jp, zp


# ---- the course evaluator (evaluate_phase7.py), functions only ----
#!/usr/bin/env python3
"""Local evaluator for CS-781 Phase 7 — Prior Case Retrieval (PCR).

It uses the same validation rules and Micro-F1@7 / Micro-Precision@7 /
Micro-Recall@7 definitions as the Codabench scorer.
"""

import argparse
import json
from pathlib import Path
from typing import Any

K = 7


def reject_constant(value: str) -> None:
    raise ValueError(f"Non-standard JSON numeric value '{value}' is not allowed.")


def pairs_no_duplicates(pairs):
    result = {}
    duplicates = []
    for key, value in pairs:
        if key in result:
            duplicates.append(key)
        result[key] = value
    if duplicates:
        raise ValueError(f"Duplicate JSON object key(s) detected: {list(dict.fromkeys(duplicates))}")
    return result


def load_json(path: Path) -> Any:
    try:
        with path.open("r", encoding="utf-8") as f:
            return json.load(f, object_pairs_hook=pairs_no_duplicates, parse_constant=reject_constant)
    except json.JSONDecodeError as exc:
        raise ValueError(
            f"Invalid JSON in {path} at line {exc.lineno}, column {exc.colno}: {exc.msg}"
        ) from exc


def load_truth(path: Path) -> dict[str, list[str]]:
    data = load_json(path)
    # Accept either the simple gold.json mapping or a Phase-7 dev query file:
    # {query_id: {text: [...], relevant_candidates: [...]}}
    if not isinstance(data, dict) or not data:
        raise ValueError("Truth must be a non-empty JSON object.")
    out: dict[str, list[str]] = {}
    for query_id, value in data.items():
        if isinstance(value, dict) and "relevant_candidates" in value:
            labels = value["relevant_candidates"]
        else:
            labels = value
        if not isinstance(query_id, str) or not query_id.strip():
            raise ValueError("Truth contains an empty/invalid query ID.")
        if not isinstance(labels, list):
            raise ValueError(f"Truth for query '{query_id}' must contain a list of relevant candidate IDs.")
        cleaned=[]; seen=set()
        for idx,candidate_id in enumerate(labels):
            if not isinstance(candidate_id,str) or not candidate_id.strip():
                # Treat blank source labels as absent labels, matching the builder.
                continue
            if candidate_id in seen:
                raise ValueError(f"Truth for query '{query_id}' contains duplicate candidate ID '{candidate_id}'.")
            seen.add(candidate_id); cleaned.append(candidate_id)
        out[query_id]=cleaned
    return out


def load_candidate_ids(path: Path) -> set[str]:
    data = load_json(path)
    if not isinstance(data, list) or not data:
        raise ValueError("candidate_ids must be a non-empty JSON list.")
    ids=[]; seen=set()
    for idx,v in enumerate(data):
        if not isinstance(v,str) or not v.strip(): raise ValueError(f"candidate_ids index {idx} is invalid.")
        if v in seen: raise ValueError(f"candidate_ids contains duplicate ID '{v}'.")
        seen.add(v); ids.append(v)
    return set(ids)


def load_prediction(path: Path) -> Any:
    return load_json(path)


def validate(pred: Any, gold: dict[str,list[str]], candidates: set[str]) -> list[str]:
    errors=[]
    if not isinstance(pred,dict): return ["prediction.json must be a JSON object mapping query IDs to ranked lists."]
    gold_ids=set(gold); pred_ids=set(pred)
    for q in sorted(gold_ids-pred_ids): errors.append(f"Missing required query ID '{q}'.")
    for q in sorted(pred_ids-gold_ids): errors.append(f"Unknown query ID '{q}'.")
    for q in sorted(gold_ids & pred_ids):
        vals=pred[q]
        if not isinstance(vals,list):
            errors.append(f"Query '{q}': prediction must be a list."); continue
        if len(vals)!=K: errors.append(f"Query '{q}': expected exactly {K} candidates, got {len(vals)}.")
        seen=set()
        for i,c in enumerate(vals):
            if not isinstance(c,str) or not c.strip(): errors.append(f"Query '{q}': index {i} is not a non-empty string."); continue
            if c in seen: errors.append(f"Query '{q}': duplicate candidate ID '{c}'.")
            seen.add(c)
            if c not in candidates: errors.append(f"Query '{q}': candidate ID '{c}' is outside the candidate pool.")
    return errors


def score(gold:dict[str,list[str]], pred:dict[str,list[str]]) -> dict[str,float|int]:
    tp=fp=fn=0
    for q,labels in gold.items():
        g=set(labels); p=set(pred[q][:K]); tp+=len(g&p); fp+=len(p-g); fn+=len(g-p)
    precision=tp/(tp+fp) if tp+fp else 0.0
    recall=tp/(tp+fn) if tp+fn else 0.0
    f1=(2*tp)/(2*tp+fp+fn) if 2*tp+fp+fn else 0.0
    return {"primary_metric_1":f1,"secondary_metric_1":precision,"secondary_metric_2":recall,"tp":tp,"fp":fp,"fn":fn}

In [3]:
FILES = ['p7res_p7_moe__moe_lorma.pt', 'p7res_p7_moe__moe_partial2.pt', 'p7res_p7_moe__moe_lorma_strip.pt', 'p7res_p7_gqa__gqa_lorma.pt', 'p7res_p7_gqa__gqa_full_ft.pt', 'p7res_p7_gqa__gqa_partial4.pt', 'p7res_p7_gqa__gqa_ia3.pt', 'p7res_p7_gqa__gqa_qlora.pt', 'p7res_p7_gdn__gdn_lorma.pt', 'p7res_p7_gdn__gdn_partial4.pt', 'p7res_p7_lstm__lstm_full_ft.pt', 'p7res_p7_lstm__lstm_partial1.pt', 'p7res_p7_kd_gqa__kd_gqa_lorma_scores.pt', 'p7res_p7_kd_gqa__kd_gqa_lorma_embed.pt', 'p7res_p7_kd_gqa__kd_gqa_lorma_both.pt', 'p7res_p7_kd_lstm__kd_lstm_full_ft.pt', 'p7res_p7_mor__mor_lorma.pt', 'p7res_p7_mor__mor_full_ft.pt', 'p7res_p7_mor__kd_mor_full_ft.pt', 'p7res_p7_moe2__moe_uni_list.pt', 'p7res_p7_moe2__moe_uni_list_proj0.pt', 'p7res_p7_moe2__moe_uni_maxsim.pt', 'p7res_p7_gqa2__gqa_uni_list.pt', 'p7res_p7_gqa2__gqa_uni_list_proj0.pt', 'p7res_p7_gqa2__gqa_uni_list_8ch.pt', 'p7res_p7_ablate_a__a_ref.pt', 'p7res_p7_ablate_a__a_prefix.pt', 'p7res_p7_ablate_a__a_prompt.pt', 'p7res_p7_ablate_a__a_obj_own.pt', 'p7res_p7_ablate_a__a_obj_listwise.pt', 'p7res_p7_ablate_a__a_obj_pairwise.pt', 'p7res_p7_ablate_a__a_obj_margin.pt', 'p7res_p7_ablate_a__a_neg_random.pt', 'p7res_p7_ablate_a__a_neg_mixed.pt', 'p7res_p7_ablate_a__a_neg_inbatch.pt', 'p7res_p7_ablate_a__a_neg_hard3.pt', 'p7res_p7_ablate_a__a_random_init.pt', 'p7res_p7_ablate_b__b_ref.pt', 'p7res_p7_ablate_b__b_pool_last.pt', 'p7res_p7_ablate_b__b_pool_meanlast.pt', 'p7res_p7_ablate_b__b_doc_max.pt', 'p7res_p7_ablate_b__b_doc_attn.pt', 'p7res_p7_ablate_b__b_doc_first.pt', 'p7res_p7_ablate_b__b_doc_maxsim.pt', 'p7res_p7_ablate_b__b_sim_dot.pt', 'p7res_p7_ablate_b__b_proj0.pt', 'p7res_p7_ablate_b__b_train1chunk.pt', 'p7res_p7_ablate_b__b_sel_uniform.pt', 'p7res_p7_ablate_b__b_sel_first.pt', 'p7res_p7_ablate_b__b_win256.pt', 'p7res_p7_ablate_b__b_overlap.pt', 'p7res_p7_ablate_b__b_strip.pt']
# =====================================================================================================
# Phase 7 fusion / ensembles (no model, CPU is enough): every finished setup's dev and test score matrices are loaded from the
# bucket (p7res_*.pt) and combined with the lexical rankers. Weights are chosen on dev (118 queries: small, so a 95% bootstrap
# interval is printed for the key comparisons). Writes submission zips for the best combinations.
# =====================================================================================================
import numpy as np, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "scikit-learn", "scipy"], check=False)
import scipy.sparse as sp
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
P7 = "Phase_7_PCR"
def load_json(p):
    with open(p, encoding="utf-8") as f: return json.load(f)
Q, CP, REL = {}, {}, {}
for s, S in (("dev", "Dev"), ("test", "Test")):
    Q[s] = {str(k): v for k, v in load_json(get_dataset_file(f"{P7}/{S}/{s}.json")).items()}
    CP[s] = {str(k): v for k, v in load_json(get_dataset_file(f"{P7}/{S}/{s}_candidates.json")).items()}
REL["dev"] = {q: [str(c) for c in v.get("relevant_candidates", []) if isinstance(c, str) and c.strip()] for q, v in Q["dev"].items()}
POOL = {s: sorted(CP[s]) for s in CP}; QIDS = {s: sorted(Q[s]) for s in Q}
FUS_DIR = f"{WORK}/p7_fusion"; os.makedirs(FUS_DIR, exist_ok=True)


def lexical(s, kind):
    q = [doc_text(Q[s][i]) for i in QIDS[s]]; c = [doc_text(CP[s][i]) for i in POOL[s]]
    if kind == "tfidf":
        tv = TfidfVectorizer(lowercase=True, token_pattern=r"(?u)\b\w+\b", ngram_range=(1, 2), sublinear_tf=True, min_df=2, max_df=0.9, dtype=np.float32)
        X = tv.fit_transform(c); return (tv.transform(q) @ X.T).toarray()
    cv = CountVectorizer(lowercase=True, token_pattern=r"(?u)\b\w+\b", dtype=np.float32)          # BM25 (k1 1.2, b 0.75), query term frequency
    X = cv.fit_transform(c).tocsr(); N = X.shape[0]; df = np.diff(X.tocsc().indptr); idf = np.log(1 + (N - df + 0.5) / (df + 0.5)).astype(np.float32)
    dl = np.asarray(X.sum(1)).ravel(); X = X.tocoo(); den = X.data + 1.2 * (0.25 + 0.75 * dl[X.row] / dl.mean())
    W = sp.csr_matrix(((X.data * 2.2 / den) * idf[X.col], (X.row, X.col)), shape=X.shape)
    return (cv.transform(q) @ W.T).toarray()
t0 = time.time()
LEX = {(s, k): lexical(s, k) for s in ("dev", "test") for k in ("tfidf", "bm25")}
print(f"lexical scores ready [{time.time() - t0:.0f}s]")

SYS = {}
for f in FILES:
    try:
        r = torch.load(fetch_model_file(f), map_location="cpu", weights_only=False)
    except Exception:
        print("  (missing)", f); continue
    assert list(r["dev_qids"]) == QIDS["dev"] and list(r["test_qids"]) == QIDS["test"]
    SYS[f.replace("p7res_", "").replace(".pt", "")] = dict(dev=r["dev_scores_dense"].astype(np.float32), test=r["test_scores_dense"].astype(np.float32), f1=r["dev"]["f1"])
print(f"{len(SYS)} trained setups loaded")


def ev(Sd):
    return retrieval_scores(REL["dev"], rank_from_scores(Sd, QIDS["dev"], POOL["dev"]))
def per_query_f1(Sd):
    rk = rank_from_scores(Sd, QIDS["dev"], POOL["dev"]); out = []
    for q in QIDS["dev"]:
        g = set(REL["dev"][q]); p = set(rk[q][:7]); tp = len(g & p); out.append((tp, 7 - tp, len(g) - tp))
    return np.array(out)
def boot(a, b, n=2000, seed=0):
    """paired bootstrap over queries of micro-F1(a) - micro-F1(b)."""
    A, B = per_query_f1(a), per_query_f1(b); rng = np.random.default_rng(seed); d = []
    for _ in range(n):
        i = rng.integers(0, len(A), len(A)); x, y = A[i].sum(0), B[i].sum(0)
        d.append(2 * x[0] / (2 * x[0] + x[1] + x[2]) - 2 * y[0] / (2 * y[0] + y[1] + y[2]))
    d = np.array(d); return float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5)), float((d <= 0).mean())
Z = zscore_rows
ROWS = []
def show(name, Sd, St=None, ref=None):
    m = ev(Sd); extra = ""
    if ref is not None:
        lo, hi, p = boot(Sd, ref); extra = f" | vs TF-IDF [{lo:+.4f}, {hi:+.4f}] p={p:.3f}"
    print(f"  {name:58s} F1@7 {m['f1']:.4f} P@7 {m['p']:.4f} R@7 {m['r']:.4f} R@100 {m['r@100']:.3f} MRR {m['mrr']:.4f}{extra}")
    ROWS.append(dict(name=name, **m, St=St))
    return m

TF, BM = LEX[("dev", "tfidf")], LEX[("dev", "bm25")]
print("\nA. Lexical rankers (no Phase-1 model)")
show("TF-IDF uni+bigram", TF); show("BM25 (query tf)", BM)
for w in (0.25, 0.5, 1.0):
    show(f"TF-IDF + {w} x BM25", Z(TF) + w * Z(BM))

print("\nB. Each trained setup: dense alone, and dense + lambda x TF-IDF (lambda on a finer grid; random-init and GDN show how much of a hybrid is TF-IDF alone)")
LAMS = [1, 2, 3, 4, 6, 8, 12, 16]
BEST_H = {}
for n, s in sorted(SYS.items(), key=lambda kv: -kv[1]["f1"]):
    res = {lam: ev(Z(s["dev"]) * 1.0 + lam * Z(TF))["f1"] for lam in LAMS}; lam = max(res, key=res.get); BEST_H[n] = (lam, res[lam])
    print(f"  {n:50s} dense {s['f1']:.4f} | hybrid by lambda " + " ".join(f"{l}:{v:.4f}" for l, v in res.items()) + f" | best {lam}: {res[lam]:.4f}")

print("\nC. Reciprocal-rank fusion (k = 60) of TF-IDF and each of the 5 best dense setups")
def rrf(*mats, k=60):
    out = 0
    for M in mats:
        r = np.argsort(np.argsort(-M, 1), 1); out = out + 1.0 / (k + r + 1)
    return out
for n, s in sorted(SYS.items(), key=lambda kv: -kv[1]["f1"])[:5]:
    show(f"RRF(TF-IDF, {n})", rrf(TF, s["dev"]))

print("\nD. Ensembles of the Phase-1 dense models only (mean of z-scored similarities; no lexical score)")
top = [n for n, _ in sorted(SYS.items(), key=lambda kv: -kv[1]["f1"])]
for k in (2, 3, 5, 8):
    if len(top) >= k:
        show(f"dense ensemble, top {k}", sum(Z(SYS[n]["dev"]) for n in top[:k]) / k, sum(Z(SYS[n]["test"]) for n in top[:k]) / k)
# greedy: add the setup that helps most, up to 6
chosen, cur, best = [], None, -1
for _ in range(6):
    cand = [(ev(sum(Z(SYS[m]["dev"]) for m in chosen + [n]))["f1"], n) for n in top if n not in chosen]
    if not cand: break
    f, n = max(cand)
    if f <= best: break
    chosen.append(n); best = f
GREEDY_DENSE = chosen
show(f"dense ensemble, greedy {chosen}", sum(Z(SYS[n]["dev"]) for n in chosen), sum(Z(SYS[n]["test"]) for n in chosen))

print("\nE. Dense ensembles + TF-IDF (weight on TF-IDF chosen on dev)")
FUSED = {}
for label, members in (("top 3", top[:3]), ("top 5", top[:5]), ("greedy dense", GREEDY_DENSE)):
    if not members: continue
    Dd = sum(Z(SYS[n]["dev"]) for n in members) / len(members); Dt = sum(Z(SYS[n]["test"]) for n in members) / len(members)
    res = {lam: ev(Z(Dd) + lam * Z(TF))["f1"] for lam in LAMS}; lam = max(res, key=res.get)
    FUSED[label] = (members, lam)
    show(f"{label} dense + {lam} x TF-IDF", Z(Dd) + lam * Z(TF), Z(Dt) + lam * Z(LEX[('test', 'tfidf')]), ref=TF)

print("\nF. Significance of the best single-model hybrids vs TF-IDF alone (paired bootstrap over the 118 dev queries)")
for n in sorted(BEST_H, key=lambda n: -BEST_H[n][1])[:5]:
    lam = BEST_H[n][0]; show(f"{n} + {lam} x TF-IDF", Z(SYS[n]["dev"]) + lam * Z(TF), Z(SYS[n]["test"]) + lam * Z(LEX[("test", "tfidf")]), ref=TF)

# ---------------- submissions: the best dense-only ensemble and the best fusion (both use the Phase-1 models)
SUBS = []
dense_rows = [r for r in ROWS if r["name"].startswith("dense ensemble") and r["St"] is not None]
fus_rows = [r for r in ROWS if r["St"] is not None and not r["name"].startswith("dense ensemble")]
for tag, rows_ in (("dense_ensemble", dense_rows), ("fusion", fus_rows)):
    if not rows_: continue
    b = max(rows_, key=lambda r: r["f1"])
    rk = rank_from_scores(b["St"], QIDS["test"], POOL["test"])
    jp, zp = write_submission(rk, QIDS["test"], set(POOL["test"]), FUS_DIR, f"p7_{tag}")
    upload_model_file(zp); SUBS.append((os.path.basename(zp), b["name"], b["f1"]))
print("\nsubmission files:")
for z, n, f in SUBS: print(f"  {z:40s} = {n} (dev F1@7 {f:.4f})")
json.dump([{k: v for k, v in r.items() if k != "St"} for r in ROWS], open(f"{FUS_DIR}/p7_fusion.json", "w"), indent=1, default=str); upload_model_file(f"{FUS_DIR}/p7_fusion.json")
print(f"done in {(time.time() - NB_START) / 60:.0f} min")

dev.json:   0%|          | 0.00/5.87M [00:00<?, ?B/s]

Phase_7_PCR/Dev/dev_candidates.json: reconstructing file:   0%|          |  0.00B / 51.9MB            

Phase_7_PCR/Dev/dev_candidates.json: downloading bytes:           |  0.00B            

test.json:   0%|          | 0.00/9.34M [00:00<?, ?B/s]

Phase_7_PCR/Test/test_candidates.json: reconstructing file:   0%|          |  0.00B / 71.1MB            

Phase_7_PCR/Test/test_candidates.json: downloading bytes:           |  0.00B            

... (11 outputs left out)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Validating:           |   0%            

uploaded p7_fusion.json to bucket kavincg23/cs781-models
done in 3 min
